# Game environment: do implied totals and gamescript predict beyond the projection?

Issue #133, under the "what is actually predictive" epic (#114). `game_environment` (#118) derives
`implied_team_total`, `implied_margin`, `gamescript_lean` and kickoff conditions (`wind`, `temp`,
`roof`) from `schedules`' betting lines and weather, for free — no new data source. Its own docstring
explicitly defers the predictive question to this ticket. This notebook runs it through
`weekly_backtest.score_signal` (#131), the harness `notebooks/defense_matchup.ipynb` (#132) already
proved out on a real multi-season sample, and answers the four questions the ticket asks in order.

In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.query import q
from src.gold.weekly_backtest import score_signal
from src.gold.league_scoring import league_points, STAT_COLUMNS

pd.set_option("display.width", 160)

## Building the three inputs `score_signal` needs

Same shape as `defense_matchup.ipynb`'s (#132): **`actuals`** is `weekly_stats`' raw counting stats
scored through `league_points`, restricted to the four positions it can score (`league_settings` has
no kicking coefficients, so K — the position the "wind hurts kickers" claim is actually about — isn't
computable here any more than it was for `defense_vs_position.py`; noted, not worked around).
**`signal`** joins a `game_environment` column onto the player's own week through `weekly_stats.team`
+ `(season, week)`. **`projection`** is built as a *left* join onto the full player-week population,
the same fix #132 found necessary and documented: `score_signal` inner-joins the projection frame
before its per-baseline `dropna`, so a bare inner join against `weekly_projections` (2026-only, zero
overlap with 2015-2025 actuals) would silently wipe out every baseline's result, not just the
projection baseline's own. No change to `weekly_backtest.py` itself was needed — the harness already
handles a padded, all-NaN-where-missing projection frame correctly, exactly as its own docstring
promises for a stricter baseline's gaps.

"Conditions" (question 3) is tested as three separate signals rather than wind alone: `wind` itself,
`temp`, and `is_sheltered` — a binary reading of `roof` (dome/closed = 1, outdoors/open = 0), since a
retractable roof closed for the game blocks wind and precipitation the same way a dome does.

One coverage note found while building the join: `weekly_stats.team` back-labels three relocated
franchises (Raiders, Chargers, Rams) to their current city for every season, while `schedules` (and
therefore `game_environment`) use the abbreviation actually in use at the time — so 2015-2016 rows
for those three teams don't join. It's a small, understood gap (counted below), unrelated to any of
the four questions, so it's left as an honest note rather than a team-code reconciliation this ticket
didn't ask for.

In [2]:
LEAGUES = q("SELECT * FROM league_settings")

STATS = q(f"""
    SELECT player_id, season, week, position, team, {", ".join(STAT_COLUMNS)}
    FROM weekly_stats
    WHERE season_type = 'REG' AND position IN ('QB', 'RB', 'WR', 'TE')
""")

GE = q("""
    SELECT season, week, team, implied_team_total, implied_margin, gamescript_lean, wind, temp, roof
    FROM game_environment
""")
# Ordinal encoding of the named bucket, so it can be scored the same way as a continuous signal —
# tests whether the bucket loses anything relative to the raw margin it's built from.
LEAN_ORDER = {"big_underdog": -2, "underdog": -1, "pick_em": 0, "favorite": 1, "big_favorite": 2}
GE["gamescript_lean_ord"] = GE["gamescript_lean"].map(LEAN_ORDER)
# A closed retractable roof shields the game from wind and precipitation the same way a dome does.
SHELTER_ORDER = {"outdoors": 0, "open": 0, "closed": 1, "dome": 1}
GE["is_sheltered"] = GE["roof"].map(SHELTER_ORDER)

LEAGUE_SCORING = {"sleeper": "half_ppr", "espn": "ppr"}
SIGNAL_COLUMNS = {
    "implied_team_total": "implied_team_total",
    "implied_margin": "implied_margin",
    "gamescript_lean_ord": "gamescript_lean_ord",
    "wind": "wind",
    "temp": "temp",
    "is_sheltered": "is_sheltered",
}


def build_actuals(league_row):
    out = STATS[["player_id", "season", "week", "position", "team"]].copy()
    out["actual_points"] = league_points(STATS, league_row)
    return out


def build_signal(actuals, column):
    merged = actuals[["player_id", "season", "week", "team"]].merge(
        GE[["season", "week", "team", column]], on=["season", "week", "team"], how="left"
    )
    return merged.rename(columns={column: "signal_value"})[
        ["player_id", "season", "week", "signal_value"]
    ]


def build_projection(league_key, population):
    real = q(
        "SELECT player_id, season, week, sleeper_points FROM weekly_projections WHERE scoring = ?",
        [LEAGUE_SCORING[league_key]],
    )
    return population[["player_id", "season", "week"]].merge(
        real, on=["player_id", "season", "week"], how="left"
    )


# Coverage note: how much of the population the team-code mismatch above actually costs, in both
# raw counts (what the docstring cites) and the ratio computed from them.
covered = STATS.merge(GE[["season", "week", "team"]], on=["season", "week", "team"])
missing_rows = len(STATS) - len(covered)
missing_rows, len(STATS), missing_rows / len(STATS)

(1383, 61977, 0.022314729657776273)

In [3]:
results = []
for _, league in LEAGUES.iterrows():
    league_key = league["league_key"]
    actuals = build_actuals(league)
    projection = build_projection(league_key, actuals)
    for signal_name, column in SIGNAL_COLUMNS.items():
        signal = build_signal(actuals, column)
        scored = score_signal(signal, actuals, projection)
        scored["league_key"] = league_key
        scored["signal_name"] = signal_name
        results.append(scored)

results = pd.concat(results, ignore_index=True)
results.shape

(132, 13)

<a id="q1"></a>
## 1. Does implied team total predict a player's points beyond the projection, per position?

Weakly, and not robustly. Holding a player's own season-to-date PPG fixed, RB (incremental rho
0.024, n = 14,346 across 181 clustered season-weeks, p = 0.004) and WR (-0.014, n = 22,800, p = 0.024
— the *opposite* sign from "more offense helps everyone") both clear significance — but neither
survives against the last-3 PPG baseline instead (RB p = 0.15, WR p = 0.37, shown further down). QB
and TE are flat and insignificant against either baseline. A finding that depends on which
walk-forward baseline holds it fixed is read as unconfirmed rather than real, the same standard
`draft_strategy.py`'s slope check applies.

In [4]:
q1 = results[
    (results["signal_name"] == "implied_team_total")
    & (results["baseline"] == "season_to_date_ppg")
    & (results["league_key"] == "sleeper")
][["position", "n", "n_weeks", "signal_rho", "incremental_rho", "p_value", "ci_low", "ci_high"]]
q1.sort_values("position").reset_index(drop=True)

,position,n,n_weeks,signal_rho,incremental_rho,p_value,ci_low,ci_high
0,ALL,54295,181,0.078639,0.001938,0.684031,-0.007442,0.011317
1,QB,5913,181,0.230824,-0.014682,0.259745,-0.040308,0.010944
2,RB,14346,181,0.076577,0.023753,0.004214,0.007583,0.039923
3,TE,11236,181,0.084343,0.011475,0.255497,-0.008374,0.031323
4,WR,22800,181,0.072377,-0.014048,0.023888,-0.026215,-0.001881


<a id="q1b"></a>
Re-running against last-3 PPG instead of season-to-date confirms the fragility — same signal, same
positions, neither baseline agreeing with the other on significance:

In [5]:
q1b = results[
    (results["signal_name"] == "implied_team_total")
    & (results["baseline"] == "last3_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] != "ALL")
][["position", "n", "n_weeks", "incremental_rho", "p_value"]]
q1b.sort_values("position").reset_index(drop=True)

,position,n,n_weeks,incremental_rho,p_value
0,QB,4532,159,0.012455,0.422755
1,RB,11420,159,0.013553,0.146514
2,TE,8790,159,0.010476,0.351728
3,WR,18331,159,-0.006739,0.367910


<a id="q2"></a>
## 2. Does implied margin / gamescript lean predict differently by position? (RB-favorite / WR-underdog)

Yes, and it confirms only half the folk model. RB (incremental rho 0.032 against season-to-date PPG,
n = 14,346 / 181 weeks, p = 0.0001; 0.024 against last-3, p = 0.007) and TE (0.021, n = 11,236,
p = 0.028; 0.023 against last-3, p = 0.037) hold up against **both** walk-forward baselines — the
more robust pattern question 1 lacked. QB and WR are flat on both (WR: -0.006, n = 22,800, p = 0.33
against season-to-date). That's the RB-on-favorites half of the prior holding and the
WR-on-underdogs half not: a favored team's back gets more work and scores more than his own recent
baseline predicts; a trailing team's receiver does not get a corresponding bump. `gamescript_
lean`'s named bucket (the ordinal encoding of the same threshold `game_environment.py` already
applies) scores almost identically to the continuous `implied_margin` it's built from — RB 0.035, TE
0.018 — losing only a little of TE's significance (p = 0.066 vs. 0.028), so the bucket is a fine
proxy for the number underneath it.

In [6]:
q2 = results[
    (results["signal_name"].isin(["implied_margin", "gamescript_lean_ord"]))
    & (results["baseline"] == "season_to_date_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] != "ALL")
][["signal_name", "position", "n", "n_weeks", "incremental_rho", "p_value", "ci_low", "ci_high"]]
q2.sort_values(["signal_name", "position"]).reset_index(drop=True)

,signal_name,position,n,n_weeks,incremental_rho,p_value,ci_low,ci_high
0,gamescript_lean_ord,QB,5913,181,-0.008089,0.506308,-0.032056,0.015879
1,gamescript_lean_ord,RB,14346,181,0.034810,0.000030,0.018763,0.050858
2,gamescript_lean_ord,TE,11236,181,0.017570,0.066120,-0.001182,0.036322
3,gamescript_lean_ord,WR,22800,181,-0.005898,0.318741,-0.017538,0.005742
4,implied_margin,QB,5913,181,-0.007188,0.552049,-0.030995,0.016618
5,implied_margin,RB,14346,181,0.031515,0.000089,0.016011,0.047019
6,implied_margin,TE,11236,181,0.021304,0.027585,0.002378,0.040230
7,implied_margin,WR,22800,181,-0.005609,0.327226,-0.016875,0.005657


In [7]:
# Robustness: does implied_margin's RB/TE effect survive against the last-3 PPG baseline too?
q2b = results[
    (results["signal_name"] == "implied_margin")
    & (results["baseline"] == "last3_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] != "ALL")
][["position", "n", "n_weeks", "incremental_rho", "p_value"]]
q2b.sort_values("position").reset_index(drop=True)

,position,n,n_weeks,incremental_rho,p_value
0,QB,4532,159,0.012661,0.369616
1,RB,11420,159,0.023798,0.006837
2,TE,8790,159,0.023317,0.037340
3,WR,18331,159,-0.001723,0.800348


<a id="q3"></a>
## 3. Do conditions matter — wind especially?

Wind and roof do; raw temperature doesn't hold up. Wind's incremental rho against season-to-date PPG
is -0.057 for QB (n = 3,904, p = 0.0015) and -0.022 for WR (n = 15,072, p = 0.011); against last-3
PPG instead, -0.057 (p = 0.0065) and -0.025 (p = 0.0082) — both positions clear significance against
both baselines, the strongest robustness pattern of anything measured here. `is_sheltered` (a closed
roof or dome) shows the mirror-image finding for WR — a positive 0.021 against season-to-date
(p = 0.002) and 0.027 against last-3 (p = 0.0002), both significant — consistent with the same
underlying effect read from the other direction, though it doesn't reach significance for QB despite
carrying the same sign (season-to-date p = 0.12, last-3 p = 0.06). `temp` looked promising for QB at
first glance (season-to-date p = 0.08, positive) but fails the same robustness check `implied_team_
total` failed: it flips to significant on last-3 PPG in one league (ESPN p = 0.043) and not the other
(Sleeper p = 0.082), and RB/TE/WR show nothing. Read as unconfirmed. RB and TE show no wind, roof, or
temperature effect on any measure. QB carries the largest confirmed effect size of any signal in this
notebook, which is the "wind hurts deep passing" half of the common claim; the kicker half can't be
checked in this warehouse (`league_points` has no kicking coefficients). This is a different weather
claim than `punt_environment.py`'s "doesn't order at all" finding — that one was specifically about
punting, not about every weather effect this warehouse could measure.

In [8]:
q3 = results[
    (results["signal_name"].isin(["wind", "temp", "is_sheltered"]))
    & (results["baseline"] == "season_to_date_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] != "ALL")
][["signal_name", "position", "n", "n_weeks", "incremental_rho", "p_value"]]
q3.sort_values(["signal_name", "position"]).reset_index(drop=True)

,signal_name,position,n,n_weeks,incremental_rho,p_value
0,is_sheltered,QB,5913,181,0.023944,0.116622
1,is_sheltered,RB,14346,181,0.006959,0.332005
2,is_sheltered,TE,11236,181,0.002173,0.837610
3,is_sheltered,WR,22800,181,0.020560,0.002375
4,temp,QB,3904,173,0.028599,0.081644
5,temp,RB,9469,173,-0.009305,0.369193
6,temp,TE,7398,173,0.014246,0.230725
7,temp,WR,15072,173,-0.000054,0.995350
8,wind,QB,3904,173,-0.057393,0.001549
9,wind,RB,9469,173,-0.017408,0.098257


In [9]:
# Robustness: the same three condition signals against last-3 PPG instead of season-to-date.
q3b = results[
    (results["signal_name"].isin(["wind", "temp", "is_sheltered"]))
    & (results["baseline"] == "last3_ppg")
    & (results["league_key"] == "sleeper")
    & (results["position"] != "ALL")
][["signal_name", "position", "n", "n_weeks", "incremental_rho", "p_value"]]
q3b.sort_values(["signal_name", "position"]).reset_index(drop=True)

,signal_name,position,n,n_weeks,incremental_rho,p_value
0,is_sheltered,QB,4532,159,0.033452,0.055047
1,is_sheltered,RB,11420,159,0.001064,0.899057
2,is_sheltered,TE,8790,159,0.012143,0.288885
3,is_sheltered,WR,18331,159,0.027090,0.000232
4,temp,QB,3032,154,0.043435,0.024843
5,temp,RB,7636,154,-0.004330,0.698893
6,temp,TE,5855,154,0.008917,0.519782
7,temp,WR,12251,154,0.004450,0.646054
8,wind,QB,3032,154,-0.056607,0.006483
9,wind,RB,7636,154,-0.007788,0.523786


<a id="q4"></a>
## 4. Is the line's value already priced into the projection?

Unanswerable, same root cause #132 hit: **n = 0** for every league, every signal, every position on
the `sleeper_points` baseline. Scoring "beyond the projection" needs a player-week where a completed
game's actual points and a Sleeper weekly projection both exist, and this warehouse has none —
`weekly_stats` runs 2015-2025, `weekly_projections.sleeper_points` only 2026 (#117). This is the
second measurement ticket in this epic to confirm the same gap independently; re-run this section
once #117's archive accumulates a few weeks of 2026.

In [10]:
q4 = results[results["baseline"] == "sleeper_points"][
    ["league_key", "signal_name", "position", "n", "n_weeks"]
]
q4["n"].eq(0).all(), q4["n_weeks"].eq(0).all()

(True, True)

## Cross-league check

The findings that held up against both walk-forward baselines above — `implied_margin`'s RB/TE
effect, `wind`'s QB/WR effect, `is_sheltered`'s WR effect — re-run on the ESPN league's own scoring
(1.0 PPR vs. Sleeper's 0.5), confirmed to within 0.002 of Spearman rho on every one of them. The
signals that *didn't* survive the baseline-swap check above (`implied_team_total`, `temp`) aren't
re-checked here — a signal already found unconfirmed on this warehouse's own history doesn't need a
second league to also fail to confirm it.

In [11]:
ROBUST = [
    ("implied_margin", "RB"), ("implied_margin", "TE"),
    ("wind", "QB"), ("wind", "WR"),
    ("is_sheltered", "WR"),
]
cross_league = results[
    (results["baseline"] == "season_to_date_ppg")
    & results.apply(lambda r: (r["signal_name"], r["position"]) in ROBUST, axis=1)
][["signal_name", "league_key", "position", "incremental_rho", "p_value"]]
cross_league.sort_values(["signal_name", "position", "league_key"]).reset_index(drop=True)

,signal_name,league_key,position,incremental_rho,p_value
0,implied_margin,espn,RB,0.031002,0.000067
1,implied_margin,sleeper,RB,0.031515,0.000089
2,implied_margin,espn,TE,0.020373,0.035833
3,implied_margin,sleeper,TE,0.021304,0.027585
4,is_sheltered,espn,WR,0.022347,0.000983
5,is_sheltered,sleeper,WR,0.020560,0.002375
6,wind,espn,QB,-0.058459,0.001372
7,wind,sleeper,QB,-0.057393,0.001549
8,wind,espn,WR,-0.023667,0.006773
9,wind,sleeper,WR,-0.022359,0.010536


## Verdict

**Display-only**, recorded in `game_environment.py`'s module docstring, `punt_environment.py`-style.
`implied_margin` / `gamescript_lean` is the strongest finding (RB, TE — robust to both walk-forward
baselines) and confirms half the RB-favorite/WR-underdog folk model, not all of it. `wind` is next
(QB, WR — also robust to both baselines) and confirms the deep-passing half of the wind claim;
`is_sheltered` finds the same effect from the other direction for WR. `implied_team_total` and `temp`
are both weak and don't survive a baseline swap, so they're read as unconfirmed rather than real.
Every confirmed number holds cross-league to within 0.002 of Spearman rho. All of it stays *display*
rather than *weighted*, the same reason #132's verdict does: the actual promotion bar — beating the
vendor's own weekly projection — is unanswerable right now (question 4), not because the raw signals
are weak. Worth re-running once #117's archive gives this a season with both actuals and a live
projection to hold fixed.